# Learn Environment Panda Robot
Author: NAME

## Controller_Task
### Position_Controller

> In this task, you will learn ...
>
> - wie du mit einem Position Controller arbeitest, um die Gelenke eines Roboters auf spezifische Positionen zu bewegen.
> - wie man den Position Controller startet
> - wie man eine Nachricht für den Position Controller erstellt
> - wie man die Zielposition sendt und die Bewegung überprüft

In [ ]:
##### DO NOT CHANGE #####

import rospy
from controller_manager_msgs.srv import LoadController, SwitchController, SwitchControllerRequest, UnloadController
from trajectory_msgs.msg import JointTrajectory, JointTrajectoryPoint

##### DO NOT CHANGE #####

Lade den Controller:

ROS verwendet den Service /controller_manager/load_controller, um Controller zu laden.
Du musst den Namen des Controllers angeben, den du laden möchtest, in diesem Fall position_joint_trajectory_controller.
Aktiviere den Controller:

Nachdem der Controller geladen ist, musst du ihn starten. Dies geschieht mit dem Service /controller_manager/switch_controller.
Du kannst mehrere Controller gleichzeitig starten und stoppen, indem du deren Namen in den Service-Request übergibst.
Verwende die folgende Vorlage:

Nutze den Service load_controller, um den gewünschten Controller zu laden.
Starte den Service switch_controller, um den Position Controller zu aktivieren.

In [ ]:
##### DO NOT CHANGE #####

import rospy
from controller_manager_msgs.srv import LoadController, SwitchController, SwitchControllerRequest, UnloadController

controller_name = 'position_joint_trajectory_controller'
conflicting_controller = 'effort_joint_trajectory_controller'

# Funktion zum Entladen eines Controllers
def unload_controller(controller_name):
    rospy.wait_for_service('/controller_manager/switch_controller')
    try:
        switch_controller = rospy.ServiceProxy('/controller_manager/switch_controller', SwitchController)
        req = SwitchControllerRequest()
        req.stop_controllers = [controller_name]
        req.strictness = SwitchControllerRequest.BEST_EFFORT
        response = switch_controller(req)
        assert response.ok, "Controller konnte nicht entladen werden."
    except rospy.ServiceException as e:
        rospy.logerr("Service call failed: %s" % e)
    rospy.wait_for_service('/controller_manager/unload_controller')
    try:
        unload_srv = rospy.ServiceProxy('/controller_manager/unload_controller', UnloadController)
        unload_resp = unload_srv(controller_name)
        assert unload_resp.ok, "Controller konnte nicht entladen werden."
    except rospy.ServiceException as e:
        rospy.logerr("Service call failed: %s" % e)

# Entladen des alten Controllers
unload_controller(conflicting_controller)

# Service für das Laden des Controllers
rospy.wait_for_service('/controller_manager/load_controller')
load_service = rospy.ServiceProxy('/controller_manager/load_controller', LoadController)
response = load_service(controller_name)
assert response.ok, "Controller konnte nicht geladen werden."

# Service für das Aktivieren des Controllers
rospy.wait_for_service('/controller_manager/switch_controller')
switch_service = rospy.ServiceProxy('/controller_manager/switch_controller', SwitchController)
switch_req = SwitchControllerRequest()
switch_req.start_controllers = [controller_name]
switch_req.strictness = SwitchControllerRequest.BEST_EFFORT
response = switch_service(switch_req)
assert response.ok, "Controller konnte nicht gestartet werden."

##### DO NOT CHANGE #####

Nachrichtentyp:

Der Position Controller empfängt Nachrichten vom Typ trajectory_msgs/JointTrajectory.
Diese Nachricht enthält:
joint_names: Eine Liste der Gelenke, die gesteuert werden sollen.
points: Eine Liste von Positionen, die die Gelenke ansteuern sollen.
Erstelle die Nachricht:

Beginne mit einer JointTrajectory-Nachricht.
Definiere die Gelenke, die du bewegen möchtest, indem du die Namen in der Liste joint_names einträgst.
Füge Zielpositionen hinzu:

Erstelle ein JointTrajectoryPoint, um die Zielpositionen für jedes Gelenk zu definieren.
Ergänze die Felder:
positions: Die Zielpositionen in Radiant (z. B. [0.5, -0.5, 0.0]).
time_from_start: Die Dauer, die der Roboter benötigt, um die Positionen zu erreichen.
Sende die Nachricht:

Veröffentliche die Nachricht auf dem Topic /position_joint_trajectory_controller/command.

In [ ]:
##### DO NOT CHANGE #####

# Initialisiere ROS
rospy.init_node('position_control', anonymous=True)

# Publisher für das Command-Topic
pub = rospy.Publisher('/position_joint_trajectory_controller/command', JointTrajectory, queue_size=10)

# Warte, bis der Publisher bereit ist
rospy.sleep(1)

# Erstelle die Nachricht
traj_msg = JointTrajectory()
traj_msg.joint_names = ['panda_joint1', 'panda_joint2', 'panda_joint3', 'panda_joint4', 'panda_joint5', 'panda_joint6', 'panda_joint7']

# Definiere die Zielpositionen
point = JointTrajectoryPoint()
point.positions = [0.5, -0.5, 0.0, 0.0, 0.5, -0.5, 0.0]  # TODO: Füge Zielpositionen (in Radiant) für jedes Gelenk hinzu
point.time_from_start = rospy.Duration(2.0)  # TODO: Lege die Zeit fest, bis die Bewegung abgeschlossen sein soll

# Füge den Punkt zur Trajektorie hinzu
traj_msg.points.append(point)

# Sende die Nachricht
pub.publish(traj_msg)
print("Zielposition wurde gesendet.")

##### DO NOT CHANGE #####

In dieser Erweiterung definieren Sie zwei Zielpositionen.
Die erste Zielposition wird erreicht, bevor der Roboter zur zweiten Zielposition übergeht.
Was du tun musst:

Füge eine zweite Zielposition zu traj_msg.points hinzu.
Die zweite Zielposition sollte eine andere Pose ansteuern, um eine sichtbare Bewegung zu demonstrieren.
Planung der Bewegung:

Verwenden Sie die Zeitsteuerung time_from_start, um sicherzustellen, dass der Roboter genug Zeit hat, die erste Position zu erreichen, bevor er zur nächsten übergeht.
Die Zeitangaben müssen aufeinander aufbauen:
Die erste Zielposition benötigt z. B. 2 Sekunden (time_from_start = 2.0).
Die zweite Zielposition wird nach weiteren 3 Sekunden erreicht (time_from_start = 5.0).
Schritte:

Erstelle eine erste Zielposition: Definiere die Gelenkwinkel und die Dauer.
Erstelle eine zweite Zielposition: Erhöhe die Zeit und ändere die Gelenkwinkel.
Sende beide Punkte: Füge die Punkte zu traj_msg.points hinzu und sende die Nachricht.

In [ ]:
##### YOUR CODE HERE #####

# Nachricht erstellen
traj_msg = JointTrajectory()
traj_msg.joint_names = ['panda_joint1', 'panda_joint2', 'panda_joint3', 'panda_joint4', 'panda_joint5', 'panda_joint6', 'panda_joint7']

# Erste Zielposition definieren
point1 = JointTrajectoryPoint()
point1.positions = [0.6, -0.6, 0.2, 0.2, 0.5, -0.5, 0.2]
point1.time_from_start = rospy.Duration(2.0)
traj_msg.points.append(point1)

# Zweite Zielposition definieren
point2 = JointTrajectoryPoint()
point2.positions = [0.7, -0.2, -0.5, -0.2, -0.5, 0.7, -0.2]
point2.time_from_start = rospy.Duration(5.0)
traj_msg.points.append(point2)

# Nachricht senden
pub.publish(traj_msg)
print("Zwei Zielpositionen wurden gesendet.")

##### YOUR CODE HERE #####

In [ ]:
##### DO NOT CHANGE #####

rospy.signal_shutdown("Task completed")
roscpp_shutdown()

##### DO NOT CHANGE #####